# 🍪 Operation: Macaroon — A Cryptographic Access Control Lab

---

> *"Cryptography is not about hiding information. It's about making trust transferable."*

---

## Your Mission Briefing

Welcome, Agent.

You've been hired as a security consultant for **Galactic Industries HQ** — a gleaming skyscraper with 40 floors of offices, server rooms, break rooms, and executive suites. The building uses a cutting-edge cryptographic key system called **Macaroons** to manage access.

The CEO just left for a two-week sailing trip and handed her **Master Key** to the Head of Security before leaving. The Head of Security needs to grant access to various employees and contractors — *but the CEO cannot be reached, and no one should ever see the Master Key again*.

How does the system let you grant a Floor Manager access to Floor 1, and then let *that* Floor Manager hand a guest a pass to just the break room — **all without copying or exposing the master key?**

That's exactly what you're going to build today.

Over the course of this lab, you will:

1. **Build** the cryptographic primitive (HMAC) that makes Macaroons possible
2. **Mint** a root Macaroon using a master secret
3. **Attenuate** it — shaving off privileges — without the original secret
4. **Verify** a chain of trust end-to-end
5. **Attack** your own system to prove it holds

Let's begin.


---

## 🔐 Stage 1: The Vault's Primitive — HMAC

### Background: Why Not Just `Hash(Key + Message)`?

Before you can build a Macaroon, you need to understand the cryptographic lock at its heart: **HMAC** (Hash-based Message Authentication Code).

Your first instinct might be: *"I'll just hash the secret key together with the message!"* Something like:

```
signature = SHA256(master_secret + token_id)
```

This seems reasonable. But it has a catastrophic flaw called a **Length Extension Attack**.

SHA-256 works by processing data in 64-byte blocks, updating an internal state as it goes. When it outputs a digest, that output **is** the internal state. An attacker who sees `SHA256(secret + message)` can compute `SHA256(secret + message + evil_extension)` without ever knowing `secret`. They just feed the known digest back in as a starting state and keep hashing!

For an access control system, this would be catastrophic — an attacker could extend a legitimate token with `admin=true` and produce a valid-looking signature.

### The Real HMAC

HMAC defeats this by applying the key **twice**, using XOR-padding to create two independent keyed layers:

$$\text{HMAC}(K, M) = H\big((K \oplus opad) \;||\; H((K \oplus ipad) \;||\; M)\big)$$

Where:
- $K$ is the key, padded to the block size (64 bytes for SHA-256)
- $opad$ = `0x5c5c5c...` (outer padding, 64 bytes)
- $ipad$ = `0x363636...` (inner padding, 64 bytes)
- $||$ means concatenation
- $\oplus$ means XOR

The inner hash protects the message. The outer hash protects the inner hash with a *different* key transformation. Length extension attacks fail because an attacker would need to extend through both layers simultaneously — which is computationally infeasible.

### 🛠️ Your Task

Implement `manual_hmac(key, message)` from scratch using only Python's `hashlib.sha256`. Follow the four steps:

1. **Prepare the key**: If it's longer than 64 bytes, hash it first. If shorter, pad it with null bytes (`\x00`) to exactly 64 bytes.
2. **Create padded keys**: XOR the prepared key with `0x5c` (opad) and `0x36` (ipad), byte by byte.
3. **Compute the inner hash**: `SHA256(i_key_pad + message)`
4. **Compute the outer hash**: `SHA256(o_key_pad + inner_hash)`

At the end, your implementation must match Python's own `hmac` module exactly.

In [ ]:
import hashlib
import hmac

def manual_hmac(key: bytes, message: bytes) -> bytes:
    """
    Implement HMAC-SHA256 manually.
    
    The full formula is:
        HMAC(K, M) = SHA256( (K ^ opad) || SHA256( (K ^ ipad) || M ) )
    
    Args:
        key:     The secret key (bytes)
        message: The message to authenticate (bytes)
    
    Returns:
        A 32-byte HMAC-SHA256 digest
    """
    BLOCK_SIZE = 64  # SHA-256 operates on 64-byte blocks

    # --- Step 1: Prepare the Key ---
    # If the key is longer than one block, hash it down to size.
    # If it's shorter, right-pad it with null bytes.

    # YOUR CODE HERE
    # Hint: use hashlib.sha256(key).digest() to shorten a long key
    # Hint: use key + b'\x00' * (BLOCK_SIZE - len(key)) to pad a short key
    raise NotImplementedError("Implement Step 1: key preparation")

    # --- Step 2: Create XOR-Padded Keys ---
    # Create o_key_pad by XORing every byte of the prepared key with 0x5c
    # Create i_key_pad by XORing every byte of the prepared key with 0x36
    # Hint: bytes(x ^ 0x5c for x in key) gives you the XOR'd bytes

    # YOUR CODE HERE
    raise NotImplementedError("Implement Step 2: XOR padding")

    # --- Step 3: Inner Hash ---
    # Compute SHA256(i_key_pad + message)
    # Hint: hashlib.sha256(data).digest() returns raw bytes

    # YOUR CODE HERE
    raise NotImplementedError("Implement Step 3: inner hash")

    # --- Step 4: Outer Hash ---
    # Compute SHA256(o_key_pad + inner_hash) and return it

    # YOUR CODE HERE
    raise NotImplementedError("Implement Step 4: outer hash")

In [ ]:
# --- Verification ---
# We test your implementation against Python's battle-tested hmac module.
# If they match, your cryptographic primitive is solid.

secret = b'super_secret_key'
msg = b'Hello Macaroons'

your_result   = manual_hmac(secret, msg)
library_result = hmac.new(secret, msg, hashlib.sha256).digest()

print(f"Your HMAC:    {your_result.hex()}")
print(f"Library HMAC: {library_result.hex()}")

assert your_result == library_result, \
    "❌ Mismatch! Check your XOR constants and the order of concatenation."

print("\n✅ Stage 1 Complete: HMAC verified. The vault primitive is ready.")

---

## 🏛️ Stage 2: The Mint — Forging the Root Macaroon

### Background: What Is a Macaroon, Exactly?

A Macaroon (named after the mathematical concept, not the cookie — though the cookie is named after the same word) is a **bearer token with embedded caveats**. Unlike a simple API key, a Macaroon is a chain of HMAC signatures that proves a token was derived from a specific root secret without ever exposing that secret.

Think of it like a wax seal on a letter. The CEO seals a letter with her ring. The Head of Security can stamp *his* seal on top of hers — the final token carries both seals, and anyone with the original wax seal mold can verify the whole chain.

### The Root Macaroon

The CEO's **Master Secret** never leaves the vault. But it can produce a **Root Signature**:

```
Signature₀ = HMAC(master_secret, token_id)
```

Here, `token_id` is a human-readable label for this key (e.g., `"Manager-Key-001"`). It acts as a public identifier — the verifier will need it later to reproduce the chain. The signature is *secret*; the token_id is not.

This `Signature₀` is the Head of Security's master credential. He can hand it to employees — but he can also *attenuate* it before doing so.

### 🛠️ Your Task

Implement `mint_macaroon(token_id, master_secret)` which produces `Signature₀` by running HMAC with the `master_secret` as the key and the `token_id` (encoded as UTF-8 bytes) as the message.

In [ ]:
def mint_macaroon(token_id: str, master_secret: bytes) -> bytes:
    """
    Create the root Macaroon signature.
    
    This is the foundational operation: binding a token identifier
    to a master secret using HMAC. The output is Signature₀.
    
    Args:
        token_id:      A human-readable label for this root token (e.g., 'Manager-Key-001')
        master_secret: The CEO's vault secret — never shared, never exposed
    
    Returns:
        Signature₀: a 32-byte root HMAC digest
    """
    # YOUR CODE HERE
    # Hint: use your manual_hmac. The key is master_secret, the message is token_id as bytes.
    raise NotImplementedError("Implement mint_macaroon")

In [ ]:
# --- Mint the Root Token ---
master_secret = b'this_is_the_backend_secret'  # Locked in the CEO's vault
token_id      = 'Manager-Key-001'               # Public label; safe to share

sig0 = mint_macaroon(token_id, master_secret)

print(f"Token ID:               {token_id}")
print(f"Root Signature (sig0):  {sig0.hex()}")
print()
print("The Head of Security now holds sig0. The CEO is already on the boat.")
print("\n✅ Stage 2 Complete: Root Macaroon minted.")

---

## 🏢 Stage 3: Attenuation — The Floor Manager Gets a Restricted Pass

### Background: Delegating Without Sharing

Here is the magic at the heart of Macaroons.

The Head of Security wants to give the Floor 1 Manager access — but **only** to Floor 1. He can't hand over `sig0` directly, because then the Manager would have master-level access. And he can't call the CEO. So what does he do?

He uses `sig0` itself as a new HMAC key, and signs a **caveat** with it:

```
Signature₁ = HMAC(sig0, "floor=1")
```

He then gives the Manager **two things**:
- The list of caveats: `["floor=1"]`
- The final signature: `sig1`

Crucially:
- **The Manager cannot reverse `sig1` to get `sig0`** — HMAC is a one-way function.
- **The Manager cannot remove `floor=1`** — without `sig0` as the key, they can't reproduce `sig1` without the caveat.
- **The Manager cannot add new caveats freely** — though interestingly, they *can* add further restrictions (more on this in Stage 4).

This is called **attenuation**: you can only ever make a token *less* powerful, never more.

### The Chain So Far

```
master_secret ──HMAC──▶ sig0  ("Manager-Key-001")
    sig0      ──HMAC──▶ sig1  ("floor=1")
```

### 🛠️ Your Task

Implement `add_caveat(previous_signature, caveat)` which takes the previous signature as the HMAC key and the caveat string as the message, producing the next signature in the chain.

In [ ]:
def add_caveat(previous_signature: bytes, caveat: str) -> bytes:
    """
    Attenuate a Macaroon by appending a caveat.
    
    This uses the previous signature as the HMAC key, binding the new
    caveat into the chain. The previous signature is effectively 'consumed'
    — the caller should only pass the new signature onward.
    
    Args:
        previous_signature: The last signature in the chain (32 bytes)
        caveat:             A restriction string, e.g. 'floor=1' or 'expires=2024-12-31'
    
    Returns:
        The next signature in the chain (32 bytes)
    """
    # YOUR CODE HERE
    # Hint: HMAC where key=previous_signature and message=caveat (as UTF-8 bytes)
    raise NotImplementedError("Implement add_caveat")

In [ ]:
# --- Attenuate the Token: Floor Manager ---
caveat1 = 'floor=1'
sig1 = add_caveat(sig0, caveat1)

print("The Head of Security attenuates the master token...")
print()
print(f"Caveat applied:          '{caveat1}'")
print(f"Previous signature (hidden from Manager): {sig0.hex()[:16]}...")
print(f"New signature (sig1):    {sig1.hex()}")
print()
print("The Manager receives: token_id='Manager-Key-001', caveats=['floor=1'], sig1")
print("The Manager does NOT receive: sig0 or master_secret.")
print("\n✅ Stage 3 Complete: Token attenuated for the Floor Manager.")

---

## 🚪 Stage 4: Deeper Attenuation — The Guest Pass

### Background: Delegation All the Way Down

Now here's where it gets interesting. The Floor Manager wants to let their visiting friend into the building — but only to the break room on Floor 1. The Manager doesn't have `sig0` or `master_secret`. All they have is:
- `sig1` (their own attenuated token)
- Their list of caveats so far: `["floor=1"]`

Can they create a *further* restricted pass? **Yes!** That's the whole point.

The Manager runs:

```
Signature₂ = HMAC(sig1, "room=breakroom")
```

They give the friend:
- Caveats: `["floor=1", "room=breakroom"]`
- Signature: `sig2`

Note that the friend's token is **strictly less powerful** than the Manager's. The friend cannot access any room other than the break room. They cannot strip out `floor=1` to roam the whole floor. And they had no part in creating `sig0` or `sig1`.

### The Full Chain

```
master_secret ──HMAC──▶ sig0  (token_id: "Manager-Key-001")
    sig0      ──HMAC──▶ sig1  (caveat: "floor=1")
    sig1      ──HMAC──▶ sig2  (caveat: "room=breakroom")
```

### 🛠️ Your Task

Use `add_caveat` to append a second caveat (`room=breakroom`) to the chain. No new function needed — just show the chain extending naturally.

In [ ]:
# --- Attenuate Further: The Guest Pass ---
caveat2 = 'room=breakroom'

# YOUR CODE HERE
# Call add_caveat with sig1 and caveat2 to produce sig2
sig2 = None  # Replace this line
raise NotImplementedError("Compute sig2 using add_caveat")

In [ ]:
# --- Print the full chain for inspection ---
print("=== The Full Macaroon Chain ===")
print()
print(f"  master_secret  ──HMAC──▶  sig0  | token_id = '{token_id}'")
print(f"  {sig0.hex()[:12]}...")
print()
print(f"  sig0           ──HMAC──▶  sig1  | caveat = '{caveat1}'")
print(f"  {sig1.hex()[:12]}...")
print()
print(f"  sig1           ──HMAC──▶  sig2  | caveat = '{caveat2}'")
print(f"  {sig2.hex()[:12]}...")
print()
print("The Guest Token (what the friend carries to the door):")
print(f"  token_id : {token_id}")
print(f"  caveats  : {[caveat1, caveat2]}")
print(f"  signature: {sig2.hex()}")
print("\n✅ Stage 4 Complete: Guest pass forged (legitimately).")

---

## 🔒 Stage 5: The Door Lock — Verification

### Background: How Does the Door Know?

The door reader has access to one thing and one thing only: the **master secret** (it's a trusted server component). When the friend taps their token, the door receives:

- `token_id`: `"Manager-Key-001"`
- `caveats`: `["floor=1", "room=breakroom"]`
- `signature`: `sig2`

The door then **replays the entire chain** from scratch:

```
expected_sig0 = HMAC(master_secret, token_id)
expected_sig1 = HMAC(expected_sig0, "floor=1")
expected_sig2 = HMAC(expected_sig1, "room=breakroom")

if expected_sig2 == provided_sig2:
    ✅ GRANT ACCESS
else:
    ❌ DENY
```

The verifier never needs to know `sig1` or `sig0`. It reconstructs them internally from `master_secret` and the caveat list. Any tampering with the caveat list — adding, removing, or reordering caveats — will produce a different final signature and the door will reject the token.

### ⚠️ Timing Attack Warning

One critical detail: **never use `==` to compare signatures**. Python's `==` short-circuits on the first mismatched byte, which leaks timing information. An attacker making millions of requests can statistically infer the correct signature one byte at a time.

Always use `hmac.compare_digest(a, b)`, which compares all bytes in constant time regardless of where they differ.

### 🛠️ Your Task

Implement `verify_macaroon(master_secret, token_id, caveats, provided_signature)` which:
1. Recomputes the chain from `master_secret` using `mint_macaroon` and `add_caveat`
2. Returns `True` if the recomputed final signature matches `provided_signature`, using `hmac.compare_digest`

In [ ]:
def verify_macaroon(
    master_secret:      bytes,
    token_id:           str,
    caveats:            list,
    provided_signature: bytes
) -> bool:
    """
    Verify a Macaroon token by replaying the full HMAC chain.
    
    The verifier (the door lock) recomputes every signature from the master_secret
    and checks whether the final result matches what the token holder presented.
    
    Args:
        master_secret:      The root secret (only the verifier has this)
        token_id:           The public label from the token
        caveats:            The list of caveats, in the exact order they were applied
        provided_signature: The signature the token holder presented at the door
    
    Returns:
        True if the chain is valid and untampered, False otherwise
    """
    # --- Step 1: Recompute the root signature ---
    # YOUR CODE HERE: use mint_macaroon to get the starting signature
    raise NotImplementedError("Step 1: Recompute root signature")

    # --- Step 2: Walk through the caveats, extending the chain ---
    # YOUR CODE HERE: iterate over caveats and call add_caveat for each
    raise NotImplementedError("Step 2: Replay caveat chain")

    # --- Step 3: Constant-time comparison ---
    # YOUR CODE HERE: return hmac.compare_digest(computed_signature, provided_signature)
    # IMPORTANT: Do NOT use '==' — that's vulnerable to timing attacks!
    raise NotImplementedError("Step 3: Constant-time comparison")

In [ ]:
# --- Test Legitimate Access ---
print("The Guest taps their token at the break room door...")
print()

is_valid = verify_macaroon(
    master_secret      = master_secret,
    token_id           = token_id,
    caveats            = [caveat1, caveat2],
    provided_signature = sig2
)

print(f"Access {'GRANTED ✅' if is_valid else 'DENIED ❌'}")
assert is_valid, "The legitimate token should verify successfully!"
print("\n✅ Stage 5 Complete: Verification system is online.")

---

## 💀 Stage 6: The Attacker Suite — Trying to Break In

### Background: The Building's Enemies

Now you switch hats. You are no longer the friendly security consultant — you're the red team. Three different adversaries each have a copy of the Guest's token:

```
token_id  = 'Manager-Key-001'
caveats   = ['floor=1', 'room=breakroom']
signature = sig2
```

Each adversary has a different plan.

---

### Adversary A: The Deletion Attack

**Goal:** Remove `floor=1` from the caveat list to gain access to all floors.

The adversary hopes the door will compute `HMAC(sig0, "room=breakroom")` instead and that it will match `sig2`.

But think about what the door actually computes:
- With the deletion: `HMAC(sig0, "room=breakroom")` → some new value, call it `sig_fake`
- `sig_fake ≠ sig2`, because `sig2 = HMAC(sig1, "room=breakroom")` and `sig1 ≠ sig0`

The adversary doesn't know `sig0` or `sig1`, so they can't produce the right signature for the modified caveat list.

---

### Adversary B: The Reordering Attack

**Goal:** Swap the order of caveats, hoping the system doesn't care about ordering.

The adversary presents `["room=breakroom", "floor=1"]` with `sig2`.

But the HMAC chain is **ordered** — each signature depends on the previous one. Computing the chain in a different order produces a completely different sequence of signatures, none of which will match `sig2`.

---

### Adversary C: The Forging Attack

**Goal:** Add `admin=true` to the caveats and produce a matching signature.

To produce a valid `sig3 = HMAC(sig2, "admin=true")`, the adversary needs `sig2` as the key. But wait — **the Guest was handed `sig2`**, right?

Actually, no. In a real Macaroon system, the bearer receives the caveat list and the *final* signature, but the door verifier only trusts signatures that *it can reproduce from the master secret*. The Guest can compute `sig3 = HMAC(sig2, "admin=true")` — but when the door re-runs the chain with caveats `["floor=1", "room=breakroom", "admin=true"]`, it will produce the same `sig3`. Does this mean Macaroons are broken?

**Not if the verifier enforces the caveats!** The Macaroon system handles integrity — it proves the chain wasn't tampered with. The **application layer** must then evaluate whether the presented caveats grant the requested action. `admin=true` is only useful if the application blindly trusts it — a properly written verifier will only honour caveats it issued itself.

For this exercise, your forging test simulates a simpler case: the adversary doesn't have `sig2` and tries to present a fake signature. This always fails.

### 🛠️ Your Task

For each attack scenario, call `verify_macaroon` with the tampered inputs and **assert that it returns `False`**. The building's security must hold.

In [ ]:
print("=" * 60)
print("RED TEAM ENGAGEMENT: Testing Attack Scenarios")
print("=" * 60)
print()

# --- Attack A: Deletion --- 
# The adversary removes 'floor=1', hoping to gain building-wide access.
# They still present sig2 as their signature.
print("⚔️  Attack A: Deletion — Remove 'floor=1' from caveats")

# YOUR CODE HERE
# Call verify_macaroon with only [caveat2] (missing caveat1) and sig2
is_valid_deletion = None  # Replace this line
raise NotImplementedError("Run Attack A")

assert not is_valid_deletion, "Deletion attack should be rejected!"
print("   Result: ACCESS DENIED ✅ (Deletion attack failed — chain broken)")
print()

In [ ]:
# --- Attack B: Reordering ---
# The adversary swaps the caveat order, hoping the door doesn't care.
print("⚔️  Attack B: Reordering — Swap caveat order to ['room=breakroom', 'floor=1']")

# YOUR CODE HERE
# Call verify_macaroon with [caveat2, caveat1] (reversed) and sig2
is_valid_reorder = None  # Replace this line
raise NotImplementedError("Run Attack B")

assert not is_valid_reorder, "Reordering attack should be rejected!"
print("   Result: ACCESS DENIED ✅ (Reorder attack failed — signature mismatch)")
print()

In [ ]:
# --- Attack C: Forging ---
# The adversary adds 'admin=true' but doesn't have a valid sig3.
# They construct a bogus signature and hope for the best.
print("⚔️  Attack C: Forging — Add 'admin=true' with a fake signature")

forged_caveat = 'admin=true'
forged_sig    = b'this_is_not_a_real_signature____'  # 32 bytes of nonsense

# YOUR CODE HERE
# Call verify_macaroon with [caveat1, caveat2, forged_caveat] and forged_sig
is_valid_forge = None  # Replace this line
raise NotImplementedError("Run Attack C")

assert not is_valid_forge, "Forging attack should be rejected!"
print("   Result: ACCESS DENIED ✅ (Forge attack failed — can't produce valid sig without sig2)")
print()

In [ ]:
print("=" * 60)
print("All adversarial tests passed.")
print("Galactic Industries HQ is secure.")
print("=" * 60)
print()
print("\n✅ Stage 6 Complete: The building held. Red team engagement finished.")

---

## 🎓 Debrief: What You've Built

Congratulations, Agent. Let's review what you implemented — and why it matters.

### The Macaroon Chain

```
master_secret
    │
    │ HMAC(master_secret, token_id)
    ▼
  sig0  ◀── root token; held by Head of Security
    │
    │ HMAC(sig0, "floor=1")
    ▼
  sig1  ◀── attenuated token; held by Floor Manager
    │
    │ HMAC(sig1, "room=breakroom")
    ▼
  sig2  ◀── guest token; held by the Friend
```

Each level of the hierarchy can **only delegate downward**. You cannot forge a signature for a level you don't have, and you cannot remove constraints once they're embedded.

### Key Properties Demonstrated

| Property | How it's enforced |
|---|---|
| **Unforgeability** | HMAC is a one-way function — you can't reverse a signature |
| **Attenuation-only** | Each level uses the previous sig as a key, so you can't widen permissions |
| **Tamper evidence** | Any change to the caveat list breaks the chain |
| **No secret exposure** | The master secret never appears in any token |
| **Timing safety** | `hmac.compare_digest` prevents side-channel attacks |

### Where Real Macaroons Go Further

Real Macaroon libraries (like [PyMacaroons](https://github.com/ecordell/pymacaroons)) also support:

- **Third-party caveats**: Caveats that must be verified by a *different* service (e.g., "only valid if the auth server confirms this session is active")
- **Discharge tokens**: Tokens issued by third parties to satisfy those caveats
- **Location hints**: Metadata pointing to where a caveat should be verified

These extensions make Macaroons suitable for distributed systems like cloud APIs, where a single backend can't verify all conditions itself.

### Further Reading

- [Macaroons: Cookies with Contextual Caveats (original paper)](https://research.google/pubs/pub41892/)
- [HMAC RFC 2104](https://www.rfc-editor.org/rfc/rfc2104)
- [Length Extension Attack (Wikipedia)](https://en.wikipedia.org/wiki/Length_extension_attack)

---

*The CEO returned from her sailing trip, reviewed the access logs, and approved of everything. The Head of Security got a raise. The Guest enjoyed the break room coffee.*

*Mission complete.* 🍪